In [1]:
from dotenv import load_dotenv
import ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama.embeddings import OllamaEmbeddings

C:\Users\Hp\AppData\Local\Temp\ipykernel_16840\1653565133.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
query = "What is Openclaw/Moltbot and what are the major security concerns regarding this tool"

In [3]:
query_embeddings = ollama.embed(
    model="embeddinggemma",
    input=query
)

In [4]:
query_embeddings["embeddings"]

[[-0.087143615,
  -0.00059480366,
  -0.024379399,
  -0.0054443325,
  0.0073499354,
  0.012523819,
  -0.053222798,
  0.047365714,
  0.026328074,
  -0.030836299,
  0.018492172,
  -0.038371887,
  -0.04450872,
  0.006214556,
  0.030355608,
  0.0067734118,
  0.02174409,
  0.0036087143,
  -0.09502098,
  -0.014735013,
  0.05563491,
  0.019885797,
  -0.0047157384,
  -0.0036924905,
  -0.0108711645,
  0.015641421,
  0.043527268,
  -0.00030309422,
  0.07172454,
  0.0010877728,
  0.04052073,
  -0.032040086,
  -0.007099576,
  0.0018174698,
  0.043860868,
  0.040792547,
  -0.016264543,
  -0.035989404,
  0.005950851,
  0.016919624,
  -0.0020901177,
  0.088429905,
  -0.017183403,
  -0.016901247,
  0.011781876,
  -0.078703254,
  -0.095602766,
  -0.047742955,
  0.016761515,
  0.033255007,
  -0.03510705,
  -0.02946349,
  -0.028983187,
  0.0820685,
  0.018976124,
  -0.0017890503,
  -0.061389405,
  -0.064264804,
  -0.021896709,
  0.030444292,
  0.023832928,
  0.02227291,
  -0.0600784,
  -0.019545985,
  0.0

In [5]:
print(len(query_embeddings["embeddings"][0]))

768


In [6]:
query_embeddings = ollama.embed(
    model ="embeddinggemma",
    input=query,
    dimensions=512
)

In [7]:
# Dimensions

print(len(query_embeddings["embeddings"][0]))

512


In [8]:
# Create the loader

loader = PyPDFLoader(file_path="../Openclaw_Research_Report.pdf")
docs = loader.load()

len(docs)

34

In [9]:
# Split the documentss

chunker = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = chunker.split_documents(docs)

len(chunks)

272

In [10]:
# embed the documents

text_documents = [doc.page_content for doc in chunks]

print(len(text_documents))

272


### What went wrong & the fix

**The error:** When we tried to embed all 272 document chunks in a single `ollama.embed()` call, it crashed with:

```
ResponseError: Post "http://127.0.0.1:49673/tokenize": ... target machine actively refused it
```

**Why it happened:** Ollama runs a small helper process in the background for each model (this is what handles tokenizing text before embedding). Sending 272 chunks in one giant request overloaded that helper process and it crashed. Once it crashed, Ollama tried to talk to it on its old port and got refused — hence the "connection refused" error. Note that this had nothing to do with our code being logically wrong — it was a resource/capacity problem.

**The fix:** Instead of sending all 272 chunks at once, we now send them in small batches (32 at a time) in a loop, and combine the results. This is much lighter on the helper process, so it doesn't crash, and it also mirrors how you'd embed documents in a real production pipeline (batching is standard practice for large inputs).

In [11]:
# Embed the documents in batches to avoid overloading the Ollama runner

EMBED_BATCH_SIZE = 32  # ollama's model runner can crash/refuse connections on large batches

document_embeddings_list = []
for i in range(0, len(text_documents), EMBED_BATCH_SIZE):
    batch = text_documents[i:i + EMBED_BATCH_SIZE]
    response = ollama.embed(
        model='embeddinggemma',
        input=batch
    )
    document_embeddings_list.extend(response["embeddings"])

print(len(document_embeddings_list))

272


## Langchain OLLAMA

In [15]:
# create the embedder

langchain_embedder = OllamaEmbeddings(
    model="embeddinggemma"
)

In [17]:
# embed documents in batches (single large request crashes Ollama's tokenizer helper process, same issue as above)

langchain_document_embeddings = []
for i in range(0, len(text_documents), EMBED_BATCH_SIZE):
    batch = text_documents[i:i + EMBED_BATCH_SIZE]
    langchain_document_embeddings.extend(langchain_embedder.embed_documents(texts=batch))

print(len(langchain_document_embeddings))

272


In [18]:
len(langchain_document_embeddings)

272

In [19]:
len(langchain_document_embeddings[0])

768